# Set up Monday's tables (week 3, day 1)

For the trainer. Run this notebook once in the class workspace before Monday, from your Git folder of the course: `week-03/setup/day-01-setup.ipynb`. The numbered steps, and what to do before and after it, are in `day-01-setup.md` beside it.

It makes the schema `returns` in the catalog `course`, and two tables in it: `return_header`, Monday's return table (40 rows), and `return_summary` (40 rows), built from it by one SQL query so that it has lineage. The rows and the SQL are in `day-01-course-data.py` beside this notebook. The course made up every row: no number or amount belongs to a real taxpayer. Running it again is safe: it makes each table again, and puts back any comment a learner saved.

Each code cell has a number on its first line. Run cells 1 to 5 in order. Cell 6 is a choice: read its first lines.

In [ ]:
# 1. Settings. Set CLASS_GROUP to the group that holds the learners. The catalog is course.
CLASS_GROUP = "account users"

import os
try:
    _path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _folder = os.path.dirname(_path if _path.startswith("/Workspace") else "/Workspace" + _path)
except NameError:  # outside Databricks: the folder the notebook runs in
    _folder = os.getcwd()
course = {"__name__": "course_data"}
exec(open(os.path.join(_folder, "day-01-course-data.py"), encoding="utf-8").read(), course)
print("The tables go in course.returns, and", CLASS_GROUP, "gets read access in cell 5.")

In [ ]:
# 2. The schema and Monday's return table, return_header, with its comments and tags.
for statement in course["header_statements"]():
    spark.sql(statement)
print("Made course.returns.return_header.")

In [ ]:
# 3. return_summary, built by one SQL query on Databricks, so Unity Catalog records its lineage.
#    The query calls income_band, a Python function registered here first: lineage may not see
#    through it, which exercise 4 asks learners to predict.
spark.udf.register("income_band", course["income_band"], "string")
for statement in course["summary_statements"]():
    spark.sql(statement)
print("Made course.returns.return_summary: non_wage_income, adjustments_amt and income_band.")

In [ ]:
# 4. Read back what the learners will see. Expected: return_header 40 rows, no comment on
#    total_tax_amt and total_income; return_summary 40 rows; the tags data_class = case_key on
#    filer_key and role = label on exam_changed_tax.
for table in ["return_header", "return_summary"]:
    count = spark.sql(f"SELECT COUNT(*) AS n FROM course.returns.{table}").first()["n"]
    empty = [r["column_name"] for r in spark.sql(
        "SELECT column_name FROM course.information_schema.columns WHERE table_schema = 'returns' "
        f"AND table_name = '{table}' AND comment IS NULL").collect()]
    print(f"{table}: {count} rows; no comment on: {', '.join(empty) or 'none'}")
for r in spark.sql("SELECT table_name, column_name, tag_name, tag_value FROM course.information_schema.column_tags "
                   "WHERE schema_name = 'returns' AND table_name = 'return_header'").collect():
    print(f"tag: {r['table_name']}.{r['column_name']} {r['tag_name']} = {r['tag_value']}")

In [ ]:
# 5. Read access for the class: see the catalog and the schema, and read every table in it.
for statement in [
    f"GRANT USE CATALOG ON CATALOG course TO `{CLASS_GROUP}`",
    f"GRANT USE SCHEMA ON SCHEMA course.returns TO `{CLASS_GROUP}`",
    f"GRANT SELECT ON SCHEMA course.returns TO `{CLASS_GROUP}`",
]:
    spark.sql(statement)
print("Granted USE CATALOG, USE SCHEMA and SELECT on course.returns to", CLASS_GROUP)

In [ ]:
# 6. A choice: run this cell only if each learner should press AI generate in exercise 3.
#    AI generate needs MODIFY on the table, and MODIFY also lets a learner save a comment or
#    change rows. Run cells 2 to 4 again after class, or after anyone saves, to put the tables back.
#    Without this cell, learners use the prepared drafts in exercise 3's drafts.md.
spark.sql(f"GRANT MODIFY ON TABLE course.returns.return_header TO `{CLASS_GROUP}`")
print("Granted MODIFY on course.returns.return_header to", CLASS_GROUP)